In [1]:
!uv pip install chromadb

Checked 1 package in 4.56s


In [2]:
!uv pip install pypdf
!uv pip install PyMuPDF

Resolved 1 package in 456ms
Prepared 1 package in 381ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 221ms
 + pypdf==6.16.1
Checked 1 package in 95ms


In [3]:
import numpy as np
import numpy as np
from review_data import make_reviews              # the 60 reviews from last class
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

ModuleNotFoundError: No module named 'review_data'

In [ ]:
chunks = [
    "the food was good and the service was fast",
    "good food and very friendly staff",
    "great service and great value for money",
    "the momo was delicious and the staff were friendly",
    "delicious food, fast service, good price",
    "excellent food and excellent service",
    "the curry was delicious and the staff was kind",
    "friendly staff and fresh food",
    "fresh ingredients and good flavour",
    "the dal bhat was delicious and hot",
    "good value and the service was quick",
    "quick service and delicious coffee",
    "the staff was friendly and the food was fresh",
    "great flavour and a clean place",
    "clean tables and very good food",
    "the thukpa was excellent and hot",
    "excellent value, the food was fresh",
    "we loved the food, service was fast",
    "the biryani was delicious and the portion was generous",
    "generous portion and good price",
    "the tea was good and the staff smiled",
    "friendly waiter and delicious pastries",
    "the food arrived hot and fresh",
    "hot food, fast service, friendly staff",
    "very good experience, we will come again",
    "the chef was great and the food was delicious",
    "great place, clean and friendly",
    "the salad was fresh and the service was good",
    "good coffee and a quick, friendly staff",
    "delicious flavour and excellent price",
]


In [5]:
import chromadb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline


# Create and FIT the model once
lsa = make_pipeline(
    TfidfVectorizer(stop_words="english"),
    TruncatedSVD(n_components=2, random_state=0),

)

chunk_embeddings = lsa.fit_transform(chunks)

print("chunk embeddings:", chunk_embeddings.shape)


# Query function
def embed(query):
    embedding = lsa.transform([query])
    print("query embedding:", embedding.shape)
    return embedding


# Chroma
client = chromadb.Client()

col = client.create_collection(
    "notes_20",  # use a new collection while debugging
    metadata={"hnsw:space": "cosine"}
)

col.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks
)


# Search
q = "staff and fresh food"

query_embedding = embed(q)

results = col.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)

print(results["documents"])

NameError: name 'chunks' is not defined

In [ ]:
results

In [ ]:


client = chromadb.Client()

col = client.get_collection("notes_20")


q = "staff and fresh food"

results = col.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)


results


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# STEP 1: THE DATABASE (CHUNKING)
# ==========================================
# Imagine this is a textbook that we chopped into 5 paragraphs (chunks).
# The computer will automatically assign them IDs: 0, 1, 2, 3, 4.
chunks = [
    "Overfitting happens when a machine learning model memorizes the training data.", # ID 0
    "A database is used to store data safely.",                                       # ID 1
    "The learning rate controls how big of a step the model takes during training.",  # ID 2
    "Python is a popular programming language.",                                      # ID 3
    "A transformer is a deep learning architecture using attention mechanisms."       # ID 4
]

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATION)
# ==========================================
# We create a machine to turn English words into math (numbers).
embedder = TfidfVectorizer()

# We pass all our chunks through the machine to create our "Database Embeddings".
database_embeddings = embedder.fit_transform(chunks)


# ==========================================
# STEP 3: THE SEARCH ENGINE (THE KITCHEN)
# ==========================================
def ranked(q):
    """Takes a question, does similarity math, and returns a sorted list of Chunk IDs."""
    
    # 1. Turn the user's question into math
    q_vec = embedder.transform([q])
    
    # 2. Compare the question to every chunk in the database (Cosine Similarity)
    scores = cosine_similarity(q_vec, database_embeddings)[0]
    
    # 3. Sort the scores from highest match to lowest match
    sorted_ids = np.argsort(-scores) 
    
    # 4. Hand back the list of IDs (e.g., [2, 0, 4, 1, 3])
    return sorted_ids.tolist()


# ==========================================
# STEP 4: THE ANSWER KEY (THE HUMAN GRADER)
# ==========================================
# A human read the chunks above and wrote down the perfect answers.
EVAL = [
    ("what is overfitting?", 0),                     # The answer is in Chunk 0
    ("what does the learning rate control?", 2),     # The answer is in Chunk 2
    ("what is a transformer?", 4)                    # The answer is in Chunk 4
]


# ==========================================
# STEP 5: THE METRICS (RECALL & MRR)
# ==========================================
def rank_of_gold(q, gold):
    """Finds what position the Search Engine put the correct answer in."""
    retrieved_list = ranked(q)
    
    # If the search engine totally failed, give it an infinite rank
    if gold not in retrieved_list:
        return float('inf')
        
    # Find the index of the gold chunk, and add 1 (because humans count from 1)
    return retrieved_list.index(gold) + 1


# --- Let's run the actual test! ---

k = 3 # We only have patience to look at the Top 3 results

# Notice the square brackets [] to create a completed list before doing the math!
recall_list = [rank_of_gold(q, gold) <= k for q, gold in EVAL]
mrr_list    = [1 / rank_of_gold(q, gold) for q, gold in EVAL]

final_recall = np.mean(recall_list)
final_mrr    = np.mean(mrr_list)

print("--- AI SEARCH ENGINE REPORT CARD ---")
print(f"Final Recall@{k} Score: {final_recall * 100}%")
print(f"Final MRR Score:      {final_mrr}")

In [ ]:
q = "what is overfitting?"

def ranked(q):
    """Takes a question, does similarity math, and returns a sorted list of Chunk IDs."""
    
    # 1. Turn the user's question into math
    q_vec = embedder.transform([q])
    
    # 2. Compare the question to every chunk in the database (Cosine Similarity)
    scores = cosine_similarity(q_vec, database_embeddings)[0]
    
    # 3. Sort the scores from highest match to lowest match
    sorted_ids = np.argsort(-scores) 
    
    # 4. Hand back the list of IDs (e.g., [2, 0, 4, 1, 3])
    return sorted_ids.tolist()


    

In [ ]:
import chromadb
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# STEP 1: THE DATA & METADATA (STICKY NOTES)
# ==========================================
chunks = [
    "Oak trees grow very tall and drop leaves.",          # ID 0
    "Pine trees keep their green needles all year.",      # ID 1
    "To get stronger, you must lift heavy weights.",      # ID 2
    "Cardio training improves your heart health.",        # ID 3
    "A transformer is an advanced AI architecture."       # ID 4
]

# We attach a "sticky note" (metadata) to every single chunk
metadatas = [
    {"topic": "trees"},       # Matches Chunk 0
    {"topic": "trees"},       # Matches Chunk 1
    {"topic": "training"},    # Matches Chunk 2
    {"topic": "training"},    # Matches Chunk 3
    {"topic": "ai"}           # Matches Chunk 4
]

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATOR)
# ==========================================
vectorizer = TfidfVectorizer()
# Convert chunks to math, and turn it into a standard dense array
chunk_embeddings = vectorizer.fit_transform(chunks).toarray() 

def embed(text):
    """Helper function to translate a single question into math"""
    return vectorizer.transform([text]).toarray()[0]


# ==========================================
# STEP 3: SETTING UP THE DATABASE
# ==========================================
client = chromadb.Client() # Opens an in-memory database

# Create a collection (like a table in a database)
col = client.create_collection(
    name="notes_2",
    metadata={"hnsw:space": "cosine"}  # Tell it to use Cosine Similarity!
)


# ==========================================
# STEP 4: INGESTION (FILLING THE CABINET)
# ==========================================
# Create ID strings: ['0', '1', '2', '3', '4']
ids = [str(i) for i in range(len(chunks))]

# Hand everything over to Chroma. It organizes it instantly.
col.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)


# ==========================================
# STEP 5: THE SEARCH (WITH A FILTER!)
# ==========================================
question = "How do I build muscle?"

# Search the database, but strictly limit it to the "training" topic
results = col.query(
    query_embeddings=[embed(question).tolist()],
    n_results=2,
    where={"topic": "training"}  # The Magic Filter!
)

print("--- SEARCH RESULTS ---")
# Chroma returns a dictionary of lists, so we grab the first list of documents [0]
for doc in results['documents'][0]:
    print(f"- {doc}")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# STEP 1: THE DATABASE
# ==========================================
documents = [
    "The quick sports car",
    "A fast food delivery vehicle",
    "The fastest racing vehicle"
]

# ==========================================
# STEP 2: THE TWO DETECTIVES
# ==========================================
# Detective Sparse: Only cares about exact letter-for-letter keyword matches.
sparse_machine = CountVectorizer()
sparse_db = sparse_machine.fit_transform(documents)

# Detective Dense: Cares about the "weight" and meaning of the words (TF-IDF).
# (In a real system, you would use SVD or an LLM Embedding here).
dense_machine = TfidfVectorizer()
dense_db = dense_machine.fit_transform(documents)


# ==========================================
# STEP 3: THE NORMALIZER (The Peacemaker)
# ==========================================
def norm(scores):
    """
    Forces all scores to live perfectly between 0.0 and 1.0.
    If the highest score is 50, it divides everything by 50 so the winner is 1.0.
    """
    max_score = np.max(scores)
    if max_score == 0:
        return scores # Prevent dividing by zero if there are no matches!
    return scores / max_score


# ==========================================
# STEP 4: THE HYBRID BOSS (The Blender)
# ==========================================
def hybrid_search(query, alpha=0.5):
    print(f"\n--- Searching for: '{query}' (Alpha: {alpha}) ---")
    
    # 1. Get raw scores from Detective Dense
    q_dense = dense_machine.transform([query])
    raw_dense_scores = cosine_similarity(q_dense, dense_db)[0]
    
    # 2. Get raw scores from Detective Sparse
    q_sparse = sparse_machine.transform([query])
    raw_sparse_scores = cosine_similarity(q_sparse, sparse_db)[0]
    
    # 3. Normalize both so they are fair (0.0 to 1.0)
    dense_clean = norm(raw_dense_scores)
    sparse_clean = norm(raw_sparse_scores)
    
    # 4. THE MAGIC BLEND FORMULA
    # alpha controls Dense, (1 - alpha) controls Sparse
    final_scores = (alpha * dense_clean) + ((1 - alpha) * sparse_clean)
    
    # 5. Sort the results from highest to lowest
    winning_ids = np.argsort(-final_scores)
    
    # Print the leaderboard
    for rank, doc_id in enumerate(winning_ids):
        score = final_scores[doc_id]
        print(f"Rank {rank+1} | Score: {score:.2f} | Doc: {documents[doc_id]}")


# ==========================================
# STEP 5: RUNNING THE SIMULATION
# ==========================================
user_query = "fast vehicle"

# Scenario A: 50/50 Blend (A balanced approach)
hybrid_search(user_query, alpha=0.5)

# Scenario B: 100% Dense (Only listen to the Meaning Detective)
hybrid_search(user_query, alpha=1.0)

# Scenario C: 100% Sparse (Only listen to the Keyword Detective)
hybrid_search(user_query, alpha=0.0)

In [ ]:
import pypdf
from openai import OpenAI
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import fitz
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

pdf_path = (r"sarathi_academy_course_aiml.pdf")

doc = fitz.open(pdf_path)
text = ""
for page in doc :
    text += page.get_text() + "\n"


chunk_size = 1000
overlap = 30

chunks = []

start = 0
while start<len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

# for i, chunk in enumerate(chunks):
#     print(f"\n------Chunk {i}------")
#     print(chunk)


In [ ]:
chunks

In [6]:
import chromadb
import json
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# STEP 1: THE DATA
# ==========================================


# ==========================================
# STEP 1.5: AUTO-METADATA (GEMINI)
# ==========================================
llm = OpenAI(
    api_key="AQ.Ab8RN6J2L4Ex8G2CpxgDjMyhQWzBjAdUodmxiU1aRIEVo5CxqQ",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

TOPICS = ["RAG", "AI_ML", "Course", "Academy"]

def generate_metadata(chunks):
    numbered = "\n\n".join(f"[{i}] {c}" for i, c in enumerate(chunks))
    resp = llm.chat.completions.create(
        model="gemini-3.6-flash",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content":
            f'Label each chunk. topic must be one of {TOPICS}.\n'
            f'Return {{"items":[{{"topic":..., "summary":"one sentence"}}]}} '
            f'— one per chunk, same order, JSON only.\n\n{numbered}'}]
    )
    items = json.loads(resp.choices[0].message.content)["items"]
    return [{"topic": it.get("topic") if it.get("topic") in TOPICS else "unknown",
             "summary": it.get("summary", "")[:300]}
            for it in items]

metadatas = generate_metadata(chunks)

# Check what Gemini actually labeled before you filter on it
for c, m in zip(chunks, metadatas):
    print(f"{m['topic']:<10} | {c[:50]}")

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATOR)
# ==========================================
vectorizer = TfidfVectorizer()
chunk_embeddings = vectorizer.fit_transform(chunks).toarray()

def embed(text):
    """Helper function to translate a single question into math"""
    return vectorizer.transform([text]).toarray()[0]

# # ==========================================
# # STEP 3: SETTING UP THE DATABASE
# # ==========================================
client = chromadb.Client()
col = client.create_collection(
    name="notes_4",
    metadata={"hnsw:space": "cosine"}
)

# ==========================================
# STEP 4: INGESTION
# ==========================================
ids = [str(i) for i in range(len(chunks))]
col.add(
    ids=ids,
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=metadatas
)
client = chromadb.Client()
col = client.get_collection("notes_4")

# ==========================================
# STEP 5: THE SEARCH (WITH A FILTER!)
# ==========================================
question = "How do i become a ai ml engineer?"


# print("--- SEARCH RESULTS ---")
# for doc in results['documents'][0]:
#     print(f"- {doc}")
# print("ran")

NameError: name 'chunks' is not defined

In [ ]:
results

In [ ]:
results['documents']

In [ ]:


def make_answer(user_query: str):


    results = col.query(
    query_embeddings=[embed(user_query).tolist()],
    n_results=2,
    where={"topic": "Academy"}
    ) 

    chunks = results['documents']
    
    SYSTEM_PROMPT = f"""

    You will be given context in chunks and user query now answer the question by reading context with factual data and relevancy. Give proper explanation along different points in bullet forms

    DO NOT USE \n for line breaking. It should be proper markdown format
    context: {chunks}
    
    
    """

    resp = llm.chat.completions.create(
        model="gemini-3.6-flash",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, 
                  {"role":"user", "content":user_query}
                 ]
        
    )

    resp = resp.choices[0].message.content

    return resp
    

In [7]:
answer = make_answer("How do i become a aiml engineer?")

NameError: name 'make_answer' is not defined

In [8]:
answer

NameError: name 'answer' is not defined

In [9]:
neat_res = json.loads(answer)

NameError: name 'answer' is not defined

In [10]:
neat_res

NameError: name 'neat_res' is not defined

Based on the provided context, here is how you can become an AI/ML Engineer by following a structured learning path:',
 'steps': ['**Master Core Skills & Tools**: Focus on gaining expertise in essential technologies including Python, scikit-learn, PyTorch, Transformers, LangChain, LangGraph, Pinecone (for RAG), Agents (Mem0, MCP), FastAPI, and Docker.',
  '**Build Deployed Portfolio Projects**: Instead of creating basic toy projects, build and deploy public, production-ready artefacts for every module to showcase your practical capabilities.',
  '**Follow a Structured Syllabus**: Stick to a clear day-by-day curriculum that clearly defines what to read, what to build, and the expected outcomes, incorporating regular revision sessions (e.g., every 5th day) to ensure continuous retention.',
  '**Participate in Mentorship & Pair-Programming**: Work in small cohorts to receive direct mentor feedback, project reviews, and engage in open classroom sessions to pair-program and solve complex challenges.',
  '**Career Preparation**: Refine your professional profile by polishing your CV and LinkedIn, participating in mock interviews, and leveraging referral networks.']

In [12]:
"""

Make a class called retriver

need 3 methods

1. chunker: takes the document and chunks it
2. embedder: takes the chunks and embed it
3. search: takes the query and do similarity search



"""

'\n\nMake a class called retriver\n\nneed 3 methods\n\n1. chunker: takes the document and chunks it\n2. embedder: takes the chunks and embed it\n3. search: takes the query and do similarity search\n\n\n\n'

In [13]:
"""

Make a neural network (class network) custom methods for training and prediction.

4 layers use sequnetial to encapulate all the layers in one attribute


task: handwritten digits 

MNIST handwritten dataset. 


"""

'\n\nMake a neural network (class network) custom methods for training and prediction.\n\n4 layers use sequnetial to encapulate all the layers in one attribute\n\n\ntask: handwritten digits \n\nMNIST handwritten dataset. \n\n\n'